# FD001 RUL Prediction

Train and evaluate the RUL prediction models. This notebook runs independently from the processed FD001 data in `data/processed/`.

In [ ]:
from pathlib import Path
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
RUL_CAP = 125

np.random.seed(RANDOM_STATE)

print("FleetGuard Phase 2 initialized.")

In [ ]:
# Colab upload option:
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    uploaded = files.upload()
    PROJECT_ROOT = Path("/content")
    DATA_DIR = PROJECT_ROOT
else:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / "data" / "processed").is_dir():
        PROJECT_ROOT = PROJECT_ROOT.parent
    DATA_DIR = PROJECT_ROOT / "data" / "processed"

train = pd.read_csv(DATA_DIR / "train_fd001.csv")
val = pd.read_csv(DATA_DIR / "validation_fd001.csv")
test = pd.read_csv(DATA_DIR / "test_fd001_with_rul.csv")

print("Training:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

assert set(train.engine_id).isdisjoint(set(val.engine_id))
print("Engine leakage check: PASSED")


## 1. Feature engineering
Raw sensor readings are useful, but their recent trends can provide additional information about engine degradation.

We'll use:
- Current sensor values.
- Rolling mean over 5, 10 and 20 cycles.
- Rolling standard deviation over 5, 10 and 20 cycles.
- Current operating settings.
- Current operating cycle.

### Select useful sensors

In [ ]:
sensor_cols = [
    f"sensor_{i:02d}" for i in range(1, 22)
]

setting_cols = [
    f"setting_{i}" for i in range(1, 4)
]

constant_sensors = [
    "sensor_01",
    "sensor_05",
    "sensor_10",
    "sensor_16",
    "sensor_18",
    "sensor_19"
]

sensor_cols = [
    c for c in sensor_cols
    if c not in constant_sensors
]

print("Selected sensors:", len(sensor_cols))
print(sensor_cols)

### Create rolling features

In [ ]:
def create_features(df):

    df = df.copy()

    df = df.sort_values(
        ["engine_id", "cycle"]
    ).reset_index(drop=True)

    feature_frames = [df]

    for window in [5, 10, 20]:

        rolling_mean = (
            df.groupby("engine_id")[sensor_cols]
            .rolling(window, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
        )

        rolling_mean.columns = [
            f"{c}_mean_{window}" for c in sensor_cols
        ]

        rolling_std = (
            df.groupby("engine_id")[sensor_cols]
            .rolling(window, min_periods=2)
            .std()
            .reset_index(level=0, drop=True)
        )

        rolling_std.columns = [
            f"{c}_std_{window}" for c in sensor_cols
        ]

        feature_frames.extend([
            rolling_mean,
            rolling_std
        ])

    result = pd.concat(feature_frames, axis=1)

    result = result.fillna(0)

    return result

### Generate training features

In [ ]:
train_features = create_features(train)

print("Training feature shape:", train_features.shape)

display(train_features.head())

### Generate truncated validation trajectories
We will simulate the situation where an engine is still operating and only its history up to a particular cycle is available.

For each validation engine, we select a cutoff corresponding to an actual remaining life between 5 and 125 cycles.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

val_truncated = []

for engine_id, group in val.groupby("engine_id"):

    group = group.sort_values("cycle")

    max_cycle = group["cycle"].max()

    max_remaining = min(RUL_CAP, max_cycle - 1)

    actual_remaining = rng.integers(
        5, max_remaining + 1
    )

    cutoff = max_cycle - actual_remaining

    partial_history = group[
        group["cycle"] <= cutoff
    ].copy()

    val_truncated.append(partial_history)

val_truncated = pd.concat(
    val_truncated,
    ignore_index=True
)

print("Original validation rows:", len(val))
print("Truncated validation rows:", len(val_truncated))
print("Validation engines:", val_truncated.engine_id.nunique())

### Extract the final observation per validation engine
The model will predict RUL using the latest available observation for each engine.

In [ ]:
val_features = create_features(val_truncated)

val_points = (
    val_features
    .sort_values(["engine_id", "cycle"])
    .groupby("engine_id")
    .tail(1)
    .reset_index(drop=True)
)

print("Validation prediction points:", len(val_points))

display(
    val_points[
        ["engine_id", "cycle", "rul_uncapped", "rul"]
    ].head()
)

###Prepare model inputs

In [ ]:
DROP_COLS = [
    "engine_id",
    "rul",
    "rul_uncapped",
    "rul_at_last_cycle"
]

feature_cols = [
    c for c in train_features.columns
    if c not in DROP_COLS
]

X_train = train_features[feature_cols]
y_train = train_features["rul"]

X_val = val_points[feature_cols]
y_val = val_points["rul_uncapped"]

print("Training X:", X_train.shape)
print("Validation X:", X_val.shape)

## Train predictive models
We'll compare three models with different learning approaches.

In [ ]:
models = {

    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ]),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=15,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

try:
    from xgboost import XGBRegressor

    models["XGBoost"] = XGBRegressor(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

except ImportError:
    print("XGBoost unavailable. Install it to include this model.")

print("Models:", list(models.keys()))

### Train and compare

In [ ]:
trained_models = {}
results = []
val_predictions = {}

for name, model in models.items():

    print(f"Training {name}...")

    model.fit(X_train, y_train)

    predictions = model.predict(X_val)

    predictions = np.clip(
        predictions, 0, RUL_CAP
    )

    mae = mean_absolute_error(y_val, predictions)

    rmse = np.sqrt(
        mean_squared_error(y_val, predictions)
    )

    trained_models[name] = model
    val_predictions[name] = predictions

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse
    })

results_df = pd.DataFrame(results).sort_values("MAE")

display(results_df)

best_model_name = results_df.iloc[0]["Model"]

print("Best validation model:", best_model_name)

## NASA scoring function
The NASA C-MAPSS scoring function penalizes RUL errors asymmetrically.

Let:
d = *Predicted RUL* - *Actual RUL*

s(d) = \begin{cases} e^{-d/13}-1, & d<0\\ e^{d/10}-1, & d\geq 0 \end{cases}



The total score is the sum across engines. Lower is better.

In [ ]:
def nasa_score(actual, predicted):

    d = predicted - actual

    scores = np.where(
        d < 0,
        np.exp(-d / 13) - 1,
        np.exp(d / 10) - 1
    )

    return np.sum(scores)


nasa_results = []

for name, predictions in val_predictions.items():

    score = nasa_score(
        y_val.to_numpy(),
        predictions
    )

    nasa_results.append({
        "Model": name,
        "NASA Score": score
    })

nasa_df = pd.DataFrame(nasa_results)

results_df = results_df.merge(
    nasa_df, on="Model"
).sort_values("MAE")

display(results_df)

Use MAE, RMSE and NASA score together when deciding which model to retain. MAE
is a useful general error measure, while the NASA score captures the asymmetric penalty.

## Validation visualizations
### Compare actual and predicted RUL

In [ ]:
best_predictions = val_predictions[best_model_name]

comparison = pd.DataFrame({
    "Actual RUL": y_val.to_numpy(),
    "Predicted RUL": best_predictions
})

plt.figure(figsize=(7, 6))

plt.scatter(
    comparison["Actual RUL"],
    comparison["Predicted RUL"]
)

plt.plot(
    [0, 125],
    [0, 125],
    linestyle="--"
)

plt.xlabel("Actual RUL")
plt.ylabel("Predicted RUL")
plt.title(f"FleetGuard: {best_model_name} Predictions")

plt.tight_layout()
plt.show()

## Prediction error distribution

In [ ]:
errors = (
    best_predictions - y_val.to_numpy()
)

plt.figure(figsize=(8, 4))

sns.histplot(errors, bins=15, kde=True)

plt.axvline(0, linestyle="--")

plt.xlabel("Prediction Error (Predicted - Actual)")
plt.title("Validation Prediction Error Distribution")

plt.tight_layout()
plt.show()

## Random Forest feature importance
This plot is available when Random Forest has been trained.

In [ ]:
if "Random Forest" in trained_models:

    rf_model = trained_models["Random Forest"]

    importance = pd.Series(
        rf_model.feature_importances_,
        index=feature_cols
    ).sort_values(ascending=False)

    plt.figure(figsize=(10, 7))

    importance.head(20).sort_values().plot(
        kind="barh"
    )

    plt.xlabel("Feature Importance")
    plt.title("Top 20 FleetGuard Features")

    plt.tight_layout()
    plt.show()

### Train selected model on all training engines

In [ ]:
from sklearn.base import clone

best_model = clone(models[best_model_name])

all_train_features = create_features(
    pd.concat([train, val], ignore_index=True)
)

X_all = all_train_features[feature_cols]
y_all = all_train_features["rul"]

best_model.fit(X_all, y_all)

print("Final model trained:", best_model_name)
print("Training engines:", train.engine_id.nunique() + val.engine_id.nunique())

### Predict official test RUL

In [ ]:
test_features = create_features(test)

test_points = (
    test_features
    .sort_values(["engine_id", "cycle"])
    .groupby("engine_id")
    .tail(1)
    .sort_values("engine_id")
    .reset_index(drop=True)
)

X_test = test_points[feature_cols]

test_predictions = best_model.predict(X_test)

test_predictions = np.clip(
    test_predictions, 0, RUL_CAP
)

test_results = pd.DataFrame({
    "engine_id": test_points["engine_id"],
    "actual_rul": test_points["rul_uncapped"],
    "predicted_rul": test_predictions
})

display(test_results.head(10))

### Evaluate official test results

In [ ]:
test_mae = mean_absolute_error(
    test_results["actual_rul"],
    test_results["predicted_rul"]
)

test_rmse = np.sqrt(
    mean_squared_error(
        test_results["actual_rul"],
        test_results["predicted_rul"]
    )
)

test_nasa = nasa_score(
    test_results["actual_rul"].to_numpy(),
    test_results["predicted_rul"].to_numpy()
)

print("OFFICIAL FD001 TEST RESULTS")
print("---------------------------")
print("Model:", best_model_name)
print("MAE:", round(test_mae, 3))
print("RMSE:", round(test_rmse, 3))
print("NASA Score:", round(test_nasa, 3))

## Prediction uncertainty
We'll estimate model disagreement using predictions from the models trained on the 80-engine training partition.

This is a simple uncertainty proxy, not a statistically calibrated confidence interval.
### Ensemble prediction disagreement

In [ ]:
test_model_predictions = {}

for name, model in trained_models.items():

    predictions = model.predict(X_test)

    test_model_predictions[name] = np.clip(
        predictions, 0, RUL_CAP
    )

prediction_matrix = np.column_stack(
    list(test_model_predictions.values())
)

test_results["ensemble_mean"] = prediction_matrix.mean(axis=1)

test_results["prediction_std"] = prediction_matrix.std(axis=1)

display(
    test_results[
        [
            "engine_id",
            "actual_rul",
            "predicted_rul",
            "ensemble_mean",
            "prediction_std"
        ]
    ].head(10)
)

### Save results and model

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"

MODEL_DIR = OUTPUT_DIR / "models"
RESULT_DIR = OUTPUT_DIR / "results"
FIGURE_DIR = OUTPUT_DIR / "figures"

for folder in [MODEL_DIR, RESULT_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Save final model
joblib.dump(
    best_model,
    MODEL_DIR / "fleetguard_fd001_model.pkl"
)

# Save validation metrics
results_df.to_csv(
    RESULT_DIR / "validation_model_comparison.csv",
    index=False
)

# Save test predictions
test_results.to_csv(
    RESULT_DIR / "fd001_test_predictions.csv",
    index=False
)

# Save validation predictions
validation_results = pd.DataFrame({
    "engine_id": val_points["engine_id"],
    "actual_rul": y_val.to_numpy()
})

for name, predictions in val_predictions.items():
    validation_results[f"{name}_prediction"] = predictions

validation_results.to_csv(
    RESULT_DIR / "validation_predictions.csv",
    index=False
)

# Save test predictions from individual models
pd.DataFrame({
    "engine_id": test_points["engine_id"],
    **test_model_predictions
}).to_csv(
    RESULT_DIR / "test_model_predictions.csv",
    index=False
)

print("Phase 2 outputs saved successfully.")

### Save final comparison plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.barplot(
    data=results_df,
    x="Model",
    y="MAE",
    ax=ax
)

ax.set_title("FleetGuard: Validation Model Comparison")
ax.set_ylabel("Mean Absolute Error")

fig.tight_layout()

fig.savefig(
    FIGURE_DIR / "model_comparison.png",
    dpi=160
)

plt.show()

print("\nPHASE 2 COMPLETED")